In [ ]:
import json
import re
from pathlib import Path
from nltk.stem import PorterStemmer
import spacy
from spacy.matcher import PhraseMatcher
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

TOP_N = 10 # Adjust as needed
LEXICAL_DENSITY_PERCENTILE = 75

# INPUTS
TXT_FILE = "Boisseau.txt" # Path to .txt file containing the corpus
PHASE1_JSON = r"Boisseau-phase1_state.json"

# OUTPUT
OUTPUT_JSON = r"informative_sentences.json"

stemmer = PorterStemmer()


# ============================================================
# LOAD SPACY
# ============================================================

print("Loading spaCy...")

nlp = spacy.load("en_core_web_sm")

print("Done.")


# ============================================================
# LOAD DATA
# ============================================================

with open(PHASE1_JSON, encoding="utf8") as f:
    phase1 = json.load(f)

with open(TXT_FILE, encoding="utf8") as f:
    text = f.read()


# ============================================================
# SENTENCE SEGMENTATION
# ============================================================

print("Segmenting corpus...")

doc = nlp(text)

sentence_dict = {}

sentence_docs = {}

for i, sent in enumerate(doc.sents):

    sentence = sent.text.strip()

    if sentence == "":
        continue

    sentence_dict[i] = sentence
    sentence_docs[i] = nlp(sentence)

print(f"{len(sentence_dict)} sentences found.")


# ============================================================
# LEXICAL DENSITY
# ============================================================

def lexical_density(doc):

    tokens = [

        t

        for t in doc

        if not t.is_space
    ]

    if len(tokens) == 0:
        return 0.0

    content = [

        t

        for t in tokens

        if (
            not t.is_stop
            and not t.is_punct
        )
    ]

    return len(content) / len(tokens)


densities = {

    idx: lexical_density(doc)

    for idx, doc in sentence_docs.items()

}

density_threshold = np.percentile(

    list(densities.values()),

    LEXICAL_DENSITY_PERCENTILE
)

print(
    f"Lexical density threshold "
    f"({LEXICAL_DENSITY_PERCENTILE}th percentile): "
    f"{density_threshold:.3f}"
)


# ============================================================
# UTILITIES
# ============================================================

def stem_matches(doc, cluster_stems):

    sentence_stems = {

        stemmer.stem(token.text.lower())

        for token in doc

        if token.is_alpha

    }

    matched = {

        stem

        for stem in cluster_stems

        if stem.rstrip("*").lower() in sentence_stems

    }

    return matched


def build_phrase_matcher(ngrams):

    matcher = PhraseMatcher(
        nlp.vocab,
        attr="LOWER"
    )

    if len(ngrams) > 0:

        patterns = [

            nlp.make_doc(x)

            for x in ngrams
        ]

        matcher.add(
            "NGRAMS",
            patterns
        )

    return matcher


from spacy.matcher import PhraseMatcher


def highlight_sentence(doc, matched_stems, matched_ngrams):
    """
    Highlights matched stems and matched n-grams using spaCy token
    indices instead of regexes.

    N-grams have priority over stems.
    """

    # ---------------------------------------------------------
    # Build phrase matcher for matched n-grams
    # ---------------------------------------------------------

    matcher = PhraseMatcher(
        nlp.vocab,
        attr="LOWER"
    )

    if matched_ngrams:

        patterns = [

            nlp.make_doc(g)

            for g in matched_ngrams

        ]

        matcher.add(
            "NGRAMS",
            patterns
        )

    matches = matcher(doc)

    # ---------------------------------------------------------
    # Token flags
    # ---------------------------------------------------------

    highlight = [

        False

        for _ in doc

    ]

    # ---------------------------------------------------------
    # Highlight complete n-grams first
    # ---------------------------------------------------------

    for _, start, end in matches:

        for i in range(start, end):

            highlight[i] = True

    # ---------------------------------------------------------
    # Highlight stems not already inside n-grams
    # ---------------------------------------------------------

    normalized_cluster = {

        stem.rstrip("*").lower()

        for stem in matched_stems

    }

    for token in doc:

        if highlight[token.i]:

            continue

        if not token.is_alpha:

            continue

        token_stem = stemmer.stem(

            token.text.lower()

        )

        if token_stem in normalized_cluster:

            highlight[token.i] = True

    # ---------------------------------------------------------
    # Reconstruct text
    # ---------------------------------------------------------

    pieces = []

    for token, flag in zip(doc, highlight):

        text = token.text

        if flag:

            text = f"****{text}****"

        pieces.append(
            text + token.whitespace_
        )

    return "".join(pieces)


# ============================================================
# CLUSTER PROCESSING
# ============================================================

print("Selecting representative sentences...")

for cluster in phase1["clusterDefs"]:

    stems = cluster.get(
        "stems",
        []
    )

    ngrams = cluster.get(
        "ngrams",
        []
    )

    matcher = build_phrase_matcher(
        ngrams
    )

    candidates = []

    for idx, sent_doc in sentence_docs.items():

        density = densities[idx]

        if density < density_threshold:
            continue

        matched_stems = stem_matches(
            sent_doc,
            stems,
        )

        matched_ngrams = set()

        matches = matcher(sent_doc)

        for match_id, start, end in matches:

            matched_ngrams.add(
                sent_doc[start:end].text
            )

        if (
            len(matched_stems) == 0
            and len(matched_ngrams) == 0
        ):
            continue

        score = density * (

            1
            + len(matched_stems)
            + len(matched_ngrams)

        )

        highlighted = highlight_sentence(
            sent_doc,
            matched_stems,
            matched_ngrams,
            )

        candidates.append(

            {

                "index": idx,

                "score": round(
                    score,
                    4,
                ),

                "lexical_density": round(
                    density,
                    4,
                ),

                "matched_stems": sorted(
                    matched_stems
                ),

                "matched_ngrams": sorted(
                    matched_ngrams
                ),

                "sentence": highlighted,

            }

        )

    candidates.sort(

        key=lambda x: x["score"],

        reverse=True,
    )

    cluster["most_insightful_sentences"] = candidates[
        :TOP_N
    ]


# ============================================================
# SAVE JSON
# ============================================================

with open(

    OUTPUT_JSON,

    "w",

    encoding="utf8",

) as f:

    json.dump(

        phase1,

        f,

        indent=4,

        ensure_ascii=False,

    )

print()
print(
    f"Saved enriched JSON to\n{OUTPUT_JSON}"
)